# DINO vs Classical Segmentation Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BASE-Laboratory/BraggTrack/blob/main/notebooks/dino_segmentation_comparison.ipynb)

This notebook runs both segmentation backends on the bundled `data/sample_operando/` scans and compares their outputs side-by-side.

| Method | How it works | Strengths |
|--------|-------------|-----------|
| **Classical** | Otsu threshold → LoG enhancement → h-maxima seeds → seeded watershed → merge nearby | Fast, interpretable, well-tuned for this beamline |
| **DINO** | DINOv3 patch features → PCA → HDBSCAN clustering → 3D slice stitching → Otsu foreground mask | Learns in feature space — should generalise across beamlines/detectors without re-tuning |

Uses the **mock** DINO backend by default (no GPU required). Set `BRAGGTRACK_DINO_BACKEND=torch` for real DINOv3 features.

## Setup

In [ ]:
import os, subprocess, sys

_ON_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG")

if _ON_COLAB:
    print("Colab detected — installing BraggTrack + sample data...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "braggtrack[notebook] @ git+https://github.com/BASE-Laboratory/BraggTrack.git",
    ])
    if not os.path.isdir("data/sample_operando"):
        subprocess.check_call([
            "git", "clone", "--depth=1", "--filter=blob:none", "--sparse",
            "https://github.com/BASE-Laboratory/BraggTrack.git", "_braggtrack_repo",
        ])
        subprocess.check_call(
            ["git", "sparse-checkout", "set", "data/sample_operando"],
            cwd="_braggtrack_repo",
        )
        os.makedirs("data", exist_ok=True)
        os.rename("_braggtrack_repo/data/sample_operando", "data/sample_operando")
        subprocess.check_call(["rm", "-rf", "_braggtrack_repo"])
    os.environ.setdefault("BRAGGTRACK_DATA_ROOT", os.path.abspath("data/sample_operando"))
    print("Done.")
else:
    print("Local environment — skipping Colab setup.")

In [ ]:
%matplotlib inline
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

from braggtrack.io import discover_operando_scans, sample_operando_root
from braggtrack.segmentation import (
    extract_instance_table,
    fill_holes_binary,
    label_projection_by_intensity,
    merge_nearby_labels,
    otsu_floor_from_mip,
    otsu_threshold,
    relabel_sequential,
    remove_small_objects,
    segment_classical,
    segment_dino,
    smooth_thresholds,
)

## 1 — Load real data

Read the largest 3D numeric dataset from each H5 file (bypasses the fixed NeXus path shortlist).

In [ ]:
def load_3d_volume(h5_path: Path) -> np.ndarray:
    """Pick the largest 3-D numeric dataset in an H5 file."""
    candidates: list[tuple[str, tuple[int, ...]]] = []
    with h5py.File(h5_path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset) and obj.ndim == 3 and np.issubdtype(obj.dtype, np.number):
                candidates.append((name, obj.shape))
        f.visititems(_visit)
        if not candidates:
            raise KeyError(f"No 3D numeric dataset in {h5_path}")
        name = max(candidates, key=lambda t: int(np.prod(t[1])))[0]
        return np.asarray(f[name][...], dtype=np.float64)

scans = discover_operando_scans(sample_operando_root())
all_volumes = [load_3d_volume(s.path) for s in scans]

for s, v in zip(scans, all_volumes):
    print(f"{s.scan_name}: shape={v.shape}  intensity=[{v.min():.0f}, {v.max():.0f}]")

## 2 — Run both segmentation methods

### Classical pipeline
Otsu → LoG → h-maxima → seeded watershed → remove small → fill holes → merge nearby → relabel.

In [ ]:
MERGE_DISTANCE = 15

def run_classical(volume: np.ndarray, threshold: float) -> np.ndarray:
    res = segment_classical(
        volume,
        threshold=threshold,
        blur_passes=1,
        h_value=0.1,
        min_seed_separation=2,
        seed_peak_fraction=0.2,
        seed_response_percentile=99.95,
    )
    labels = remove_small_objects(res.labeled_volume, min_size=8)
    binary = fill_holes_binary(labels > 0)
    labels = np.where(binary, labels, 0)
    labels = merge_nearby_labels(labels, volume, max_centroid_distance=MERGE_DISTANCE)
    return relabel_sequential(labels)

raw_thresholds = [otsu_threshold(v.ravel()) for v in all_volumes]
smoothed = smooth_thresholds(raw_thresholds, window=5)

classical_labels = []
for s, v, thr in zip(scans, all_volumes, smoothed):
    lab = run_classical(v, float(thr))
    classical_labels.append(lab)
    print(f"{s.scan_name} classical: threshold={thr:.1f}, {int(lab.max())} spots")

### DINO pipeline
DINOv3 patch features → PCA → HDBSCAN → upsample → 3D stitch → Otsu foreground mask → post-process.

The post-processing (remove small, fill holes, merge nearby, relabel) is identical to keep the comparison fair.

In [ ]:
def run_dino(volume: np.ndarray) -> np.ndarray:
    res = segment_dino(volume, backend="mock")
    labels = remove_small_objects(res.labeled_volume, min_size=8)
    binary = fill_holes_binary(labels > 0)
    labels = np.where(binary, labels, 0)
    labels = merge_nearby_labels(labels, volume, max_centroid_distance=MERGE_DISTANCE)
    return relabel_sequential(labels)

dino_labels = []
for s, v in zip(scans, all_volumes):
    lab = run_dino(v)
    dino_labels.append(lab)
    print(f"{s.scan_name} DINO:      {int(lab.max())} spots")

## 3 — Spot count comparison

In [ ]:
scan_names = [s.scan_name for s in scans]
classical_counts = [int(l.max()) for l in classical_labels]
dino_counts = [int(l.max()) for l in dino_labels]

comparison_df = pd.DataFrame({
    "scan": scan_names,
    "classical_spots": classical_counts,
    "dino_spots": dino_counts,
})
comparison_df["difference"] = comparison_df["dino_spots"] - comparison_df["classical_spots"]
print(comparison_df.to_string(index=False))

print(f"\nClassical spread (max-min): {max(classical_counts) - min(classical_counts)}")
print(f"DINO spread (max-min):      {max(dino_counts) - min(dino_counts)}")

In [ ]:
x = np.arange(len(scans))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width / 2, classical_counts, width, label="Classical", color="#1f77b4")
bars2 = ax.bar(x + width / 2, dino_counts, width, label="DINO (mock)", color="#ff7f0e")
ax.set_xlabel("Scan")
ax.set_ylabel("Spot count")
ax.set_title("Spot counts: Classical vs DINO")
ax.set_xticks(x)
ax.set_xticklabels(scan_names)
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 4 — Visual comparison: tri-axis label projections

Side-by-side label overlays for each scan, projected along all three physical axes (μ, χ, d). Each row is a scan; left column = classical, right column = DINO.

In [ ]:
# Build a shared colormap large enough for both methods.
max_labels = max(
    max(int(l.max()) for l in classical_labels),
    max(int(l.max()) for l in dino_labels),
) + 1
rng_cm = np.random.RandomState(42)
label_colors = np.zeros((max_labels, 4))
label_colors[0] = [0, 0, 0, 0]
for i in range(1, max_labels):
    label_colors[i] = [*rng_cm.uniform(0.2, 0.95, 3), 0.65]
label_cmap = ListedColormap(label_colors)

axis_info = [
    (0, "MIP along mu", "chi", "d"),
    (1, "MIP along chi", "d", "mu"),
    (2, "MIP along d", "chi", "mu"),
]

fig, axes = plt.subplots(len(scans), 6, figsize=(22, len(scans) * 3.5))

for row, (s, v, c_lab, d_lab) in enumerate(zip(scans, all_volumes, classical_labels, dino_labels)):
    for col_offset, (method_name, labels) in enumerate([("Classical", c_lab), ("DINO", d_lab)]):
        for ax_idx, (axis_id, title, xlabel, ylabel) in enumerate(axis_info):
            ax = axes[row, col_offset * 3 + ax_idx]
            mip = v.max(axis=axis_id)
            floor = otsu_floor_from_mip(v, axis=axis_id)
            proj_l = label_projection_by_intensity(v, labels, axis=axis_id, mip_floor=floor)

            vlo, vhi = np.percentile(mip, [1, 99.9])
            ax.imshow(mip, cmap="gray", vmin=vlo, vmax=vhi)
            mask = np.ma.masked_where(proj_l == 0, proj_l)
            ax.imshow(mask, cmap=label_cmap, interpolation="nearest", vmin=0, vmax=max_labels - 1)

            if row == 0:
                ax.set_title(f"{method_name}\n{title}", fontsize=9)
            ax.tick_params(labelsize=6)
            if ax_idx == 0 and col_offset == 0:
                n_c = int(c_lab.max())
                n_d = int(d_lab.max())
                ax.set_ylabel(f"{s.scan_name}\nC={n_c} D={n_d}", fontsize=9)

plt.suptitle("Classical (left 3 cols) vs DINO (right 3 cols) — tri-axis label projection", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 5 — Instance feature comparison

Compare the per-spot properties (voxel count, integrated intensity, centroid, eigenvalues) between the two methods.

In [ ]:
classical_features = [extract_instance_table(l, v) for l, v in zip(classical_labels, all_volumes)]
dino_features = [extract_instance_table(l, v) for l, v in zip(dino_labels, all_volumes)]

for s, cf, df in zip(scans, classical_features, dino_features):
    print(f"\n{s.scan_name}:")
    c_df = pd.DataFrame(cf)
    d_df = pd.DataFrame(df)
    print(f"  Classical: {len(c_df)} spots, "
          f"mean voxels={c_df['voxel_count'].mean():.1f}, "
          f"total intensity={c_df['integrated_intensity'].sum():.0f}")
    print(f"  DINO:      {len(d_df)} spots, "
          f"mean voxels={d_df['voxel_count'].mean():.1f}, "
          f"total intensity={d_df['integrated_intensity'].sum():.0f}")

In [ ]:
# Voxel count and intensity distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for scan_idx, (s, cf, df) in enumerate(zip(scans, classical_features, dino_features)):
    c_vox = [r["voxel_count"] for r in cf]
    d_vox = [r["voxel_count"] for r in df]
    axes[0].scatter([scan_idx - 0.1] * len(c_vox), c_vox, c="#1f77b4", alpha=0.6, s=40,
                    label="Classical" if scan_idx == 0 else None)
    axes[0].scatter([scan_idx + 0.1] * len(d_vox), d_vox, c="#ff7f0e", alpha=0.6, s=40,
                    label="DINO" if scan_idx == 0 else None)

    c_int = [r["integrated_intensity"] for r in cf]
    d_int = [r["integrated_intensity"] for r in df]
    axes[1].scatter([scan_idx - 0.1] * len(c_int), c_int, c="#1f77b4", alpha=0.6, s=40)
    axes[1].scatter([scan_idx + 0.1] * len(d_int), d_int, c="#ff7f0e", alpha=0.6, s=40)

axes[0].set_ylabel("Voxel count")
axes[0].set_title("Voxel count per spot")
axes[0].legend()
axes[1].set_ylabel("Integrated intensity")
axes[1].set_title("Integrated intensity per spot")
for ax in axes:
    ax.set_xticks(range(len(scans)))
    ax.set_xticklabels([s.scan_name for s in scans])
    ax.set_xlabel("Scan")
plt.tight_layout()
plt.show()

## 6 — Spatial overlap (Dice coefficient)

For each scan, compute the Dice coefficient between the binary foreground masks produced by the two methods. This measures how much the methods agree on *where* spots are, regardless of how they partition them into instances.

In [ ]:
def dice(a: np.ndarray, b: np.ndarray) -> float:
    a_bool = a > 0
    b_bool = b > 0
    intersection = np.count_nonzero(a_bool & b_bool)
    total = np.count_nonzero(a_bool) + np.count_nonzero(b_bool)
    return 2.0 * intersection / total if total > 0 else 1.0

for s, c_lab, d_lab in zip(scans, classical_labels, dino_labels):
    d = dice(c_lab, d_lab)
    c_fg = np.count_nonzero(c_lab > 0)
    d_fg = np.count_nonzero(d_lab > 0)
    print(f"{s.scan_name}: Dice={d:.3f}  (classical fg={c_fg} voxels, DINO fg={d_fg} voxels)")

## 7 — Centroid scatter: classical vs DINO

Plot the centroids from both methods on the same axes. Matching centroids (spots found by both methods) will overlap; method-unique detections will stand alone.

In [ ]:
fig, axes = plt.subplots(1, len(scans), figsize=(5 * len(scans), 4.5))
if len(scans) == 1:
    axes = [axes]

for ax, s, v, cf, df in zip(axes, scans, all_volumes, classical_features, dino_features):
    mip = v.max(axis=0)
    vlo, vhi = np.percentile(mip, [1, 99.9])
    ax.imshow(mip, cmap="gray", vmin=vlo, vmax=vhi)

    for r in cf:
        ax.plot(r["centroid_chi"], r["centroid_d"], "o", mfc="none",
                mec="#1f77b4", mew=1.5, ms=12)
    for r in df:
        ax.plot(r["centroid_chi"], r["centroid_d"], "x",
                mec="#ff7f0e", mew=1.5, ms=10)

    ax.set_title(f"{s.scan_name}\nblue O = classical, orange X = DINO")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8 — Consistency across scans

A key motivation for DINO-based segmentation is consistency: the same physical spots should produce the same segmentation across consecutive scans. Compare the coefficient of variation (std/mean) of spot counts across scans for each method.

In [ ]:
c_arr = np.array(classical_counts, dtype=float)
d_arr = np.array(dino_counts, dtype=float)

c_cv = c_arr.std() / c_arr.mean() if c_arr.mean() > 0 else 0
d_cv = d_arr.std() / d_arr.mean() if d_arr.mean() > 0 else 0

consistency = pd.DataFrame({
    "Method": ["Classical", "DINO (mock)"],
    "Mean spots": [c_arr.mean(), d_arr.mean()],
    "Std spots": [c_arr.std(), d_arr.std()],
    "CV (std/mean)": [c_cv, d_cv],
    "Spread (max-min)": [c_arr.max() - c_arr.min(), d_arr.max() - d_arr.min()],
})
print(consistency.to_string(index=False))

## 9 — Per-scan feature tables

Full feature tables for both methods on scan 1, for detailed inspection.

In [ ]:
cols = ["label", "voxel_count", "integrated_intensity",
        "centroid_mu", "centroid_chi", "centroid_d",
        "eig_1", "eig_2", "eig_3"]

print("=== Classical — scan0001 ===")
display(pd.DataFrame(classical_features[0])[cols]) if classical_features[0] else print("(no spots)")

print("\n=== DINO — scan0001 ===")
display(pd.DataFrame(dino_features[0])[cols]) if dino_features[0] else print("(no spots)")

## 10 — Notes and next steps

**What the mock backend shows:** The mock DINO backend produces hash-based random features, so the clustering is not semantically meaningful — it tests the *pipeline plumbing* (slice extraction → PCA → HDBSCAN → stitching → foreground mask) but not the quality of learned representations.

**What changes with real DINOv3 weights:**
- Set `BRAGGTRACK_DINO_BACKEND=torch` (requires `torch` + `transformers` + GPU)
- The encoder extracts genuine patch-level features where similar textures cluster together
- Expect better instance separation without hand-tuned LoG/watershed parameters
- The same model should work across different beamlines and detectors

**CLI equivalent:**
```bash
# Classical
braggtrack-segment-dataset --method classical --outdir artifacts/classical

# DINO (mock)
braggtrack-segment-dataset --method dino --dino-backend mock --outdir artifacts/dino

# DINO (real weights)
braggtrack-segment-dataset --method dino --dino-backend torch --outdir artifacts/dino_real
```